In [ ]:
# AKI LLM Importance Score Collection - Prompt Generates Probabilities
#
# Queries an LLM to score each feature's clinical relevance for predicting
# postoperative AKI (creatinine ratio at hour 60) in cardiac surgery patients.
# Features are scored in batches using structured JSON outputs
# (Pydantic schema). Results are cached incrementally and exported as a CSV
# compatible with the LSP analysis pipeline.
#
# Requirements: openai, pydantic, pandas, numpy

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
path = "/content/drive/MyDrive/t60_reg_data.csv"
df = pd.read_csv(path)

In [ ]:
import os
import json
import math
import time
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI


# ------------------------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/t60_reg_data.csv"
ID_COLS = ["pat_id"]
TARGET_COL = "creatinine_ratio"

df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
print("n_features =", len(FEATURE_COLS))


# ------------------------------------------------------------------------------
# Prompt Construction
# ------------------------------------------------------------------------------
DATASET_BACKGROUND = """
We have adult patients, 80 years and older, who underwent cardiac surgery.
For each patient, we have perioperative and ICU EMR-derived features:
demographics, comorbidities, vitals/hemodynamics, ventilator settings,
fluid balance, labs, procedures, and medication doses. Features are collected up until 36 hours post-surgery.
The modeling goal is sparse linear regression to predict the change in postoperative
serum creatinine after 60 hours. Specifically, we want to predict the ratio of postoperative
serum creatinine after 60 hours over the patient's baseline (pre-surgery) creatinine levels.
""".strip()

OUTCOME_NAME = "creatinine_ratio"

def infer_feature_type(series: pd.Series) -> str:
    """Light heuristic for the LLM (helps it reason about binary flags vs continuous)."""
    if pd.api.types.is_bool_dtype(series):
        return "binary"
    if pd.api.types.is_numeric_dtype(series):
        vals = series.dropna().unique()
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            return "binary"
        return "numeric"
    return "categorical_or_text"

def build_prompt_for_batch(feature_names: List[str]) -> str:
    # Include a tiny bit of metadata that is cheap but useful:
    payload = []
    for i, name in enumerate(feature_names):
        ftype = infer_feature_type(df[name])
        payload.append({"id": i, "name": name, "type": ftype})

    features_json = json.dumps(payload, indent=2)

    prompt = f"""
You are a cardiothoracic ICU clinician and a biostatistician.

Background:
{DATASET_BACKGROUND}

Task:
We are building a sparse linear regression model to predict Acute Kidney Injury (AKI) in adult patients,
80 years and older, following cardiac surgery. The exact target variable, {OUTCOME_NAME}, is the
patient's postoperative serum creatinine at Hour 60 divided by their baseline (pre-surgery) creatinine.

For each feature, measure its clinical relevance in predicting {OUTCOME_NAME} in post-cardiac surgery patients on an inclusion probability
scale (0-1). Base your judgment on typical knowledge about kidney perfusion/hemodynamics, AKI risk factors, nephrotoxic medications,
and kidney-related labs.

Our sparse linear model will use your feature inclusion probabilities to inform a statistical model. To ensure a sparse model,
please give most features small inclusion probabilities (less than 0.05).

Rules for Aggregated Vitals and Labs:
You will evaluate many features ending in _min, _max, _mean, and _measured representing a 12-hour post-surgery window.
You must differentiate your scores based on these specific aggregations:

Collinearity Rule: We want to remove collinearity in the aggregated features. When making your choices,
you must select the aggregation that best captures the true pathology. For example, if a feature is a _mean,
and you know the _max or _min captures the true pathology better, strictly cap the feature at a score of 0.05 to
prevent redundancy.

Extremes (_max, _min): These capture acute physiological insults. If the extreme state is a known physiological trigger or direct
marker for AKI, score it according to the rubric.
Averages (_mean): These smooth out acute results. Unless the 12-hour
sustained average is the primary drive of the pathology, reduce your score.
Measurement Frequency (_measured): This represents the number of times a test or measurement was ordered. This is a behavioral proxy for\
clinical suspicion/acuity, NOT a physiological mechanism. You must score _measured features less than 0.05 unless you have strong belief that
this test or measurement indicates a physician's belief that AKI may be imminent.

Rule for Temporal Proximity (Time Epochs):
Many of the features are divided into three consecutive 12-hour epochs (0_12h, 13_24h, 25_36h) to predict a clinical outcome at hour 60.
You must explicitly adjust your scored based on the epoch:

0_12h (Initial Period): This period captures the immediate trauma of cardiac surgery, anesthesia, and the
cardiopulmonary bypass machine. Derangements are common and transient. Be very conservative with your scores
in this period. Maximum Score = 0.10.

13_24h (Trajectory Phase): This period shows whether the patient is stabilizing or deteriorating. Reduce your scores in this period.
Maximum Score = 0.20.

25_36h (Leading Indicator): This is the most critical window as it is the closest physiological snapshot to the Hour 60 outcome.
Full scoring allowed.

Scoring rubric:
Give features an inclusion probability from 0 to 1. You should sort features according to these criteria.

  (No/Weak Evidence): The feature has no meaningful physiological link to AKI,
  OR it represents routine, standard-of-care ICU maintenance that provides no
  specific prognostic value for renal failure. These features should be capped at 0.01
  (Plausible Indirect Link): There is a theoretical or indirect physiological link, but
  it is not a primary driver or established predictor of AKI. These features should be capped at 0.05.
  (Established Risk Factor): The feature is a known comorbidity, standard hemodynamic indicator,
  or medication that routinely influences renal perfusion or AKI risk, but is not definitive on its own. These features should be capped at 0.10.
  (Strong Direct Predictor): Strong clinical evidence links this feature directly to subsequent AKI. These features should be capped at 0.20.
  (Definitive/Direct Biomarker): The feature is a direct, early measurement of renal failure or
  severe hemodynamic collapse explicitly known to cause renal tubular necrosis. These features should be given larger inclusion probabilities.

Input features (JSON list):
{features_json}


Return a JSON object with key "scores" containing a list of objects.
Each object must include:
- id (same as input id)
- name (copy exactly)
- importance (probability between 0 and 1)
- reason (1–2 concise sentences)
""".strip()

    return prompt


# ------------------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------------------
class ScoreItem(BaseModel):
    id: int
    name: str
    importance: float = Field(ge=0.0, le=1.0)
    reason: str

class ScoreBatch(BaseModel):
    scores: List[ScoreItem]


# ------------------------------------------------------------------------------
# LLM API Calls
# ------------------------------------------------------------------------------

client = OpenAI(api_key="API KEY HERE")

def call_llm_for_batch(prompt: str, model: str = "gpt-5.2") -> ScoreBatch:
    """
    Uses Responses API structured parsing, so you don't need to manually parse JSON text.
    """
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": "You output only structured JSON that matches the schema."},
            {"role": "user", "content": prompt},
        ],
        text_format=ScoreBatch,
        temperature=0,
        max_output_tokens=2000,
    )
    return resp.output_parsed


def get_llm_importance_scores(
    feature_names: List[str],
    batch_size: Optional[int] = None,
    model: str = "gpt-5.2",
    sleep_s: float = 0.25,
    max_retries: int = 6,
    cache_path: str = "aki_llm_scores_probabilities.json",
) -> Dict[str, Dict]:
    """
    Returns:
      dict[feature_name] = {"importance": dbl(0,1), "reason": str}
    Caches incrementally so you can resume.
    """

    # default: ceil(sqrt(p)) like the paper’s heuristic
    if batch_size is None:
        batch_size = int(math.ceil(math.sqrt(len(feature_names))))

    # load cache if exists
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            results = json.load(f)
        print(f"[cache] loaded {len(results)} scores from {cache_path}")
    else:
        results = {}

    remaining = [f for f in feature_names if f not in results]
    print(f"Remaining features to score: {len(remaining)} (batch_size={batch_size})")

    for start in range(0, len(remaining), batch_size):
        batch = remaining[start : start + batch_size]
        prompt = build_prompt_for_batch(batch)

        # retry loop (simple exponential backoff)
        attempt = 0
        while True:
            try:
                parsed = call_llm_for_batch(prompt, model=model)
                # map back
                for item in parsed.scores:
                    results[item.name] = {
                        "importance": float(item.importance),
                        "reason": item.reason,
                    }
                # write cache
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)
                print(f"[ok] scored batch {start//batch_size + 1} / {math.ceil(len(remaining)/batch_size)}")
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {e}") from e
                backoff = (2 ** attempt) * 0.5
                print(f"[retry {attempt}] error={e} | sleeping {backoff:.1f}s")
                time.sleep(backoff)

        time.sleep(sleep_s)

    return results

# ------------------------------------------------------------------------------
# Run and Export
# ------------------------------------------------------------------------------

MODEL = "gpt-5.2"

raw = get_llm_importance_scores(
    FEATURE_COLS,
    batch_size=16,          # defaults to ceil(sqrt(p))
    model=MODEL,
    cache_path="aki_llm_scores_probabilities.json",
)

# Save raw weights (0-1)
raw_df = pd.DataFrame({
    "value": list(raw.keys()),
    "importance": [raw[k]["importance"] for k in raw],
    "reason": [raw[k]["reason"] for k in raw],
})
raw_df.to_csv("aki_weights_probabilities.csv", index=False)

n_features = 1790
Remaining features to score: 1790 (batch_size=16)
[ok] scored batch 1 / 112
[ok] scored batch 2 / 112
[ok] scored batch 3 / 112
[ok] scored batch 4 / 112
[ok] scored batch 5 / 112
[ok] scored batch 6 / 112
[ok] scored batch 7 / 112
[ok] scored batch 8 / 112
[ok] scored batch 9 / 112
[ok] scored batch 10 / 112
[ok] scored batch 11 / 112
[ok] scored batch 12 / 112
[ok] scored batch 13 / 112
[ok] scored batch 14 / 112
[ok] scored batch 15 / 112
[ok] scored batch 16 / 112
[ok] scored batch 17 / 112
[ok] scored batch 18 / 112
[ok] scored batch 19 / 112
[ok] scored batch 20 / 112
[ok] scored batch 21 / 112
[ok] scored batch 22 / 112
[ok] scored batch 23 / 112
[ok] scored batch 24 / 112
[ok] scored batch 25 / 112
[ok] scored batch 26 / 112
[ok] scored batch 27 / 112
[ok] scored batch 28 / 112
[ok] scored batch 29 / 112
[ok] scored batch 30 / 112
[ok] scored batch 31 / 112
[ok] scored batch 32 / 112
[ok] scored batch 33 / 112
[ok] scored batch 34 / 112
[ok] scored batch 35 / 

In [ ]:
from google.colab import files
files.download("aki_weights_probabilities.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_new = pd.read_csv("aki_weights_probabilities.csv")
df_new["importance"].value_counts()

,count
importance,
0.010,551
0.020,326
0.030,271
0.005,137
0.040,123
0.050,110
0.080,59
0.060,56
0.120,31
